# EchoNet-Dynamic: Storage-Safe Fine-Grained EF Experiments (Kaggle)

This notebook samples **9,500 videos** from EchoNet-Dynamic while preserving the complete official validation and test splits. It extracts **16 evenly spaced frames per video directly from the source videos** and stores only those frames as JPEGs. It does **not** write every decoded frame as PNG, because a full-frame PNG archive previously used about 19.95 GiB and would leave too little room on Kaggle working storage.

Experiments:
1. Baseline: MobileNetV3-Small + BiLSTM EF regression
2. A: regression + fine-grained EF classification
3. B: temporal attention + EF regression
4. AB: attention + classification + EF regression

**Important:** this notebook assumes the Kaggle EchoNet-Dynamic dataset is added to the notebook via *Add Input*. If the dataset path differs, set `DATASET_ROOT` manually in Cell 2. Enable Internet in Kaggle settings if ImageNet pretrained weights need to be downloaded. A model result is not guaranteed to improve merely because an extra head or attention is added.

In [ ]:
# CELL 1 — Imports and reproducibility
import os, gc, json, random, time, shutil, math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from scipy.stats import pearsonr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    print('WARNING: Kaggle GPU is not enabled. Turn on GPU in Notebook Settings.')
print('PyTorch:', torch.__version__)
print('OpenCV:', cv2.__version__)

In [ ]:
# CELL 2 — Locate the dataset and configure paths
# Kaggle inputs are read-only under /kaggle/input; outputs belong in /kaggle/working.
KAGGLE_INPUT = Path('/kaggle/input')

# If automatic detection picks the wrong folder, set this to the folder containing
# FileList.csv and Videos/, for example /kaggle/input/echonet-dynamic/EchoNet-Dynamic
DATASET_ROOT = None

if DATASET_ROOT is None:
    filelists = list(KAGGLE_INPUT.rglob('FileList.csv'))
    if not filelists:
        raise FileNotFoundError(
            'FileList.csv was not found under /kaggle/input. Add the EchoNet-Dynamic dataset '
            'to this Kaggle notebook, then rerun this cell.'
        )
    # Prefer a FileList.csv whose parent has a Videos directory
    preferred = [p for p in filelists if (p.parent / 'Videos').is_dir()]
    FILELIST_PATH = preferred[0] if preferred else filelists[0]
    DATASET_ROOT = FILELIST_PATH.parent
else:
    DATASET_ROOT = Path(DATASET_ROOT)
    FILELIST_PATH = DATASET_ROOT / 'FileList.csv'

DATASET_ROOT = Path(DATASET_ROOT)
VIDEO_DIR = DATASET_ROOT / 'Videos'
if not FILELIST_PATH.exists():
    raise FileNotFoundError(f'FileList.csv not found: {FILELIST_PATH}')
if not VIDEO_DIR.exists():
    # Search for a Videos folder under the detected input root
    matches = list(DATASET_ROOT.rglob('Videos'))
    if not matches:
        raise FileNotFoundError(f'Videos folder not found under {DATASET_ROOT}')
    VIDEO_DIR = matches[0]

WORK_ROOT = Path('/kaggle/working/echonet_fg_work')
SEQUENCES_DIR = WORK_ROOT / 'sampled_frames'
OUTPUT_DIR = Path('/kaggle/working/echonet_fg_results')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SEQUENCES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATASET_ROOT:', DATASET_ROOT)
print('FILELIST_PATH:', FILELIST_PATH)
print('VIDEO_DIR:', VIDEO_DIR)
print('SEQUENCES_DIR:', SEQUENCES_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Free disk GB:', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

In [ ]:
# CELL 3 — Read labels and match video files
file_df = pd.read_csv(FILELIST_PATH)
required = ['FileName', 'EF', 'Split']
missing = [c for c in required if c not in file_df.columns]
if missing:
    raise ValueError(f'FileList.csv is missing required columns: {missing}')

file_df = file_df.copy()
file_df['FileName'] = file_df['FileName'].astype(str).str.strip()
file_df['video_id'] = file_df['FileName'].map(lambda x: Path(x).stem)
file_df['EF'] = pd.to_numeric(file_df['EF'], errors='coerce')
file_df['Split'] = file_df['Split'].astype(str).str.upper().str.strip()
file_df['Split'] = file_df['Split'].replace({'VALIDATION': 'VAL', 'VALID': 'VAL'})
file_df = file_df.dropna(subset=['EF']).copy()
file_df = file_df[file_df['Split'].isin(['TRAIN', 'VAL', 'TEST'])].copy()
file_df = file_df[file_df['EF'].between(0, 100)].drop_duplicates('video_id')

# Build a video lookup by stem. FileList filenames typically use .avi.
video_extensions = {'.avi', '.mp4', '.mov', '.mkv', '.mpeg', '.mpg', '.wmv'}
video_lookup = {}
for p in VIDEO_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in video_extensions:
        video_lookup[p.stem] = p

file_df['video_path'] = file_df['video_id'].map(lambda vid: str(video_lookup[vid]) if vid in video_lookup else None)
matched = file_df[file_df['video_path'].notna()].copy()
missing_video_ids = file_df.loc[file_df['video_path'].isna(), 'video_id'].tolist()
print('FileList labeled videos:', len(file_df))
print('Videos matched to files:', len(matched))
print('Unmatched rows:', len(missing_video_ids))
print('Matched split counts:\n', matched['Split'].value_counts())
if len(matched) < 9500:
    raise RuntimeError(f'Only {len(matched)} videos matched; cannot select 9,500.')
if not {'TRAIN','VAL','TEST'}.issubset(set(matched['Split'])):
    raise RuntimeError('TRAIN, VAL and TEST must all be present.')

# Save mapping diagnostics
pd.DataFrame({'unmatched_video_id': missing_video_ids}).to_csv(OUTPUT_DIR / 'unmatched_videos.csv', index=False)
matched[['video_id','FileName','video_path','EF','Split']].to_csv(OUTPUT_DIR / 'matched_video_inventory.csv', index=False)


In [ ]:
# CELL 4 — Select exactly 9,500 videos without dropping official validation/test videos
# The official EchoNet-Dynamic split is preserved. All matched VAL and TEST videos are kept;
# only TRAIN is randomly subsampled if needed. This is safer than randomly sampling all splits.
TARGET_VIDEOS = 9500
rng = np.random.default_rng(SEED)
train_all = matched[matched['Split'] == 'TRAIN'].copy()
val_all = matched[matched['Split'] == 'VAL'].copy()
test_all = matched[matched['Split'] == 'TEST'].copy()

fixed_count = len(val_all) + len(test_all)
train_needed = TARGET_VIDEOS - fixed_count
if train_needed <= 0:
    raise RuntimeError(f'VAL+TEST already have {fixed_count} videos; cannot select {TARGET_VIDEOS} with this strategy.')
if train_needed > len(train_all):
    raise RuntimeError(f'Need {train_needed} train videos but only {len(train_all)} are available.')

if train_needed < len(train_all):
    train_selected = train_all.sample(n=train_needed, random_state=SEED)
else:
    train_selected = train_all

selected_df = pd.concat([train_selected, val_all, test_all], ignore_index=True)
selected_df = selected_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
assert len(selected_df) == TARGET_VIDEOS
assert selected_df['video_id'].is_unique

# Preserve all test and validation video IDs; no video may occur in multiple splits.
assert set(selected_df[selected_df.Split=='TRAIN'].video_id).isdisjoint(set(selected_df[selected_df.Split=='VAL'].video_id))
assert set(selected_df[selected_df.Split=='TRAIN'].video_id).isdisjoint(set(selected_df[selected_df.Split=='TEST'].video_id))
assert set(selected_df[selected_df.Split=='VAL'].video_id).isdisjoint(set(selected_df[selected_df.Split=='TEST'].video_id))

selection_path = OUTPUT_DIR / 'selected_9500_video_manifest.csv'
selected_df[['video_id','FileName','video_path','EF','Split']].to_csv(selection_path, index=False)
print('Selected videos:', len(selected_df))
print('Selected split counts:\n', selected_df['Split'].value_counts())
print('Saved:', selection_path)
print('All available test videos retained:', len(test_all) == len(selected_df[selected_df.Split=='TEST']))
print('All available validation videos retained:', len(val_all) == len(selected_df[selected_df.Split=='VAL']))

## Frame extraction strategy

The supplied utility writes every decoded frame as PNG. Here we keep its central idea (decode original videos and preserve temporal coverage) but only save **16 evenly spaced frames per selected video**, because the model itself uses 16 frames and Kaggle working storage is limited. We save JPEGs at quality 92 to reduce storage. No random frame is selected after the split; frame positions are deterministic and evenly spaced.

In [ ]:
# CELL 5 — Extract 16 evenly spaced frames directly from each source video
NUM_FRAMES = 16
JPEG_QUALITY = 92

# Resume-safe: if all expected JPEGs exist for a video, skip re-extracting it.
def extract_sampled_frames(video_path, video_id, output_root, num_frames=NUM_FRAMES):
    out_dir = Path(output_root) / video_id
    out_dir.mkdir(parents=True, exist_ok=True)
    expected = [out_dir / f'frame_{i:02d}.jpg' for i in range(num_frames)]
    if all(p.exists() and p.stat().st_size > 0 for p in expected):
        return {'video_id': video_id, 'success': True, 'frames_saved': num_frames, 'error': ''}

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return {'video_id': video_id, 'success': False, 'frames_saved': 0, 'error': 'Unable to open video'}

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return {'video_id': video_id, 'success': False, 'frames_saved': 0, 'error': 'Invalid frame count'}

    indices = np.linspace(0, frame_count - 1, num_frames).round().astype(int)
    saved = 0
    try:
        for seq_pos, frame_idx in enumerate(indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            ok, frame = cap.read()
            if not ok or frame is None:
                # Seek fallback: sequential decode from start if random seek failed
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                current = -1
                frame = None
                while current < int(frame_idx):
                    ok, candidate = cap.read()
                    current += 1
                    if not ok:
                        break
                    if current == int(frame_idx):
                        frame = candidate
                if frame is None:
                    raise RuntimeError(f'Could not decode frame {frame_idx}')
            out_path = out_dir / f'frame_{seq_pos:02d}.jpg'
            ok = cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            if not ok:
                raise RuntimeError(f'Failed to write {out_path}')
            saved += 1
    except Exception as e:
        cap.release()
        return {'video_id': video_id, 'success': False, 'frames_saved': saved, 'error': str(e)}
    cap.release()
    return {'video_id': video_id, 'success': saved == num_frames, 'frames_saved': saved,
            'error': '' if saved == num_frames else f'Expected {num_frames}, saved {saved}'}

# Check the available disk space before extraction.
free_gb = shutil.disk_usage('/kaggle/working').free / 1024**3
print(f'Free disk before extraction: {free_gb:.2f} GiB')
if free_gb < 3.0:
    raise RuntimeError('Less than 3 GiB free before extraction. Clear unneeded /kaggle/working files first.')

extraction_log_path = OUTPUT_DIR / 'frame_extraction_log.csv'
if extraction_log_path.exists():
    extraction_log = pd.read_csv(extraction_log_path)
    completed_ids = set(extraction_log.loc[extraction_log.success == True, 'video_id'].astype(str))
else:
    extraction_log = pd.DataFrame(columns=['video_id','success','frames_saved','error'])
    completed_ids = set()

# Retry all not-yet-successful videos; existing complete frame folders are skipped by the function.
log_by_id = {str(r.video_id): r._asdict() for r in extraction_log.itertuples(index=False)}
for row_idx, row in enumerate(tqdm(selected_df.itertuples(index=False), total=len(selected_df), desc='Extracting sampled frames'), start=1):
    vid = str(row.video_id)
    if vid in completed_ids:
        # Verify actual files still exist before trusting old log
        if all((SEQUENCES_DIR / vid / f'frame_{j:02d}.jpg').exists() for j in range(NUM_FRAMES)):
            continue
    result = extract_sampled_frames(row.video_path, vid, SEQUENCES_DIR)
    log_by_id[vid] = result
    if row_idx % 25 == 0:
        pd.DataFrame(log_by_id.values()).to_csv(extraction_log_path, index=False)

extraction_log = pd.DataFrame(log_by_id.values())
extraction_log.to_csv(extraction_log_path, index=False)
print('Extraction results:\n', extraction_log['success'].value_counts(dropna=False))
print('Failed examples:\n', extraction_log[extraction_log.success != True].head(10))
print('Free disk after extraction:', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2), 'GiB')

successful_ids = set(extraction_log.loc[extraction_log.success == True, 'video_id'].astype(str))
usable_df = selected_df[selected_df.video_id.astype(str).isin(successful_ids)].copy()
print('Videos with complete sampled sequences:', len(usable_df))
if len(usable_df) < 9000:
    raise RuntimeError('Fewer than 9,000 videos have complete sampled sequences. Inspect frame_extraction_log.csv before training.')

In [ ]:
# CELL 6 — Create the sequence manifest and verify all 16 frames per video
manifest = usable_df[['video_id','EF','Split','video_path']].copy()
manifest['sequence_dir'] = manifest['video_id'].map(lambda vid: str(SEQUENCES_DIR / str(vid)))
manifest['num_frames'] = NUM_FRAMES

# Verify exact frame count and image readability for a sample
for row in manifest.sample(min(20, len(manifest)), random_state=SEED).itertuples(index=False):
    files = sorted(Path(row.sequence_dir).glob('frame_*.jpg'))
    assert len(files) == NUM_FRAMES, f'{row.video_id}: expected {NUM_FRAMES}, got {len(files)}'
    with Image.open(files[0]) as im:
        im.verify()

MANIFEST_PATH = OUTPUT_DIR / 'sequence_manifest_9500.csv'
manifest.to_csv(MANIFEST_PATH, index=False)
print('Saved manifest:', MANIFEST_PATH)
print('Manifest rows:', len(manifest))
print('Split counts:\n', manifest['Split'].value_counts())
print('Frame files:', len(manifest) * NUM_FRAMES)
print('Free disk:', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2), 'GiB')

In [ ]:
# CELL 7 — Build PyTorch datasets and loaders
IMAGE_SIZE = 160
BATCH_SIZE = 4
NUM_WORKERS = 2

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

class EchoSequenceDataset(Dataset):
    def __init__(self, frame_df):
        self.df = frame_df.reset_index(drop=True).copy()
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        imgs = []
        for j in range(NUM_FRAMES):
            p = Path(row['sequence_dir']) / f'frame_{j:02d}.jpg'
            with Image.open(p) as im:
                imgs.append(transform(im.convert('RGB')))
        frames = torch.stack(imgs, dim=0)  # [T,C,H,W]
        ef = float(row['EF'])
        # Derived classification labels; categories are EF<40, 40<=EF<=50, EF>50.
        cls = 0 if ef < 40 else (1 if ef <= 50 else 2)
        return frames, torch.tensor(ef, dtype=torch.float32), torch.tensor(cls, dtype=torch.long), str(row['video_id'])

train_df = manifest[manifest.Split == 'TRAIN'].copy()
val_df = manifest[manifest.Split == 'VAL'].copy()
test_df = manifest[manifest.Split == 'TEST'].copy()
assert len(train_df) and len(val_df) and len(test_df)
assert set(train_df.video_id).isdisjoint(set(val_df.video_id))
assert set(train_df.video_id).isdisjoint(set(test_df.video_id))
assert set(val_df.video_id).isdisjoint(set(test_df.video_id))

train_ds, val_ds, test_ds = map(EchoSequenceDataset, [train_df, val_df, test_df])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE.type=='cuda'), persistent_workers=(NUM_WORKERS>0))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(DEVICE.type=='cuda'), persistent_workers=(NUM_WORKERS>0))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=(DEVICE.type=='cuda'), persistent_workers=(NUM_WORKERS>0))

x, y, c, ids = next(iter(train_loader))
print('Batch frames:', tuple(x.shape), 'EF:', tuple(y.shape), 'classes:', c.tolist())
print('Train/Val/Test:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# CELL 8 — Model variants: baseline, fine-grained classification, temporal attention, combined
class EchoFineGrainedModel(nn.Module):
    def __init__(self, use_classification=False, use_attention=False, hidden_size=128, dropout=0.30,
                 pretrained=True):
        super().__init__()
        self.use_classification = use_classification
        self.use_attention = use_attention
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        backbone = models.mobilenet_v3_small(weights=weights)
        self.cnn = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_dim = backbone.classifier[0].in_features  # 576 for MobileNetV3-Small
        self.lstm = nn.LSTM(self.feature_dim, hidden_size, num_layers=1,
                            batch_first=True, bidirectional=True)
        out_dim = hidden_size * 2
        if use_attention:
            self.attention = nn.Sequential(nn.Linear(out_dim, 64), nn.Tanh(), nn.Linear(64, 1))
        self.dropout = nn.Dropout(dropout)
        self.ef_head = nn.Sequential(nn.Linear(out_dim, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        if use_classification:
            self.class_head = nn.Sequential(nn.Linear(out_dim, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 3))

    def forward(self, x):
        # x [B,T,C,H,W]
        b, t, c, h, w = x.shape
        z = x.reshape(b*t, c, h, w)
        z = self.cnn(z)
        z = self.pool(z).flatten(1).reshape(b, t, self.feature_dim)
        sequence, _ = self.lstm(z)
        if self.use_attention:
            weights = torch.softmax(self.attention(sequence).squeeze(-1), dim=1)
            representation = torch.sum(sequence * weights.unsqueeze(-1), dim=1)
        else:
            representation = sequence.mean(dim=1)
        representation = self.dropout(representation)
        out = {'ef': self.ef_head(representation).squeeze(1)}
        if self.use_classification:
            out['class_logits'] = self.class_head(representation)
        return out

print('Model class defined.')

In [ ]:
# CELL 9 — Training function (best checkpoint selected by validation MAE)
EPOCHS = 12
PATIENCE = 4
LR_CNN = 1e-5
LR_HEAD = 1e-4
CLASS_LOSS_WEIGHT = 0.20
USE_PRETRAINED = True  # set False if Kaggle Internet is disabled and weights cannot be downloaded


def predict_loader(model, loader, with_classes=False):
    model.eval()
    ys, ps, classes_true, classes_pred, vids = [], [], [], [], []
    with torch.no_grad():
        for frames, ef, cls, video_ids in loader:
            frames = frames.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type=='cuda')):
                out = model(frames)
            ys.extend(ef.numpy().tolist())
            ps.extend(out['ef'].float().cpu().numpy().tolist())
            vids.extend(list(video_ids))
            if 'class_logits' in out:
                classes_true.extend(cls.numpy().tolist())
                classes_pred.extend(out['class_logits'].argmax(1).cpu().numpy().tolist())
    return np.asarray(ys), np.asarray(ps), vids, classes_true, classes_pred


def train_experiment(experiment_name, use_classification=False, use_attention=False):
    print('\n' + '='*70 + '\n', experiment_name, '\n' + '='*70)
    model = EchoFineGrainedModel(use_classification, use_attention, pretrained=USE_PRETRAINED).to(DEVICE)
    optimizer = torch.optim.AdamW([
        {'params': model.cnn.parameters(), 'lr': LR_CNN},
        {'params': list(model.lstm.parameters()) + list(model.ef_head.parameters()) +
                   (list(model.class_head.parameters()) if use_classification else []) +
                   (list(model.attention.parameters()) if use_attention else []), 'lr': LR_HEAD}
    ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    reg_loss_fn = nn.SmoothL1Loss(beta=1.0)
    cls_loss_fn = nn.CrossEntropyLoss()
    amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    best_mae, best_epoch, stale = float('inf'), 0, 0
    history = []
    ckpt_path = OUTPUT_DIR / f'{experiment_name}_best.pt'

    for epoch in range(1, EPOCHS+1):
        start = time.time()
        model.train()
        loss_sum, n_seen = 0.0, 0
        for frames, ef, cls, _ in tqdm(train_loader, desc=f'{experiment_name} epoch {epoch}'):
            frames = frames.to(DEVICE, non_blocking=True)
            ef = ef.to(DEVICE, non_blocking=True)
            cls = cls.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=amp):
                out = model(frames)
                loss = reg_loss_fn(out['ef'], ef)
                if use_classification:
                    loss = loss + CLASS_LOSS_WEIGHT * cls_loss_fn(out['class_logits'], cls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(ef)
            n_seen += len(ef)

        val_y, val_p, _, _, _ = predict_loader(model, val_loader)
        val_mae = mean_absolute_error(val_y, val_p)
        train_loss = loss_sum / max(n_seen, 1)
        scheduler.step(val_mae)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_mae': val_mae,
                        'lr_cnn': optimizer.param_groups[0]['lr'], 'lr_head': optimizer.param_groups[1]['lr']})
        print(f'Epoch {epoch}/{EPOCHS} | loss={train_loss:.4f} | val MAE={val_mae:.4f} | {time.time()-start:.1f}s')
        if val_mae < best_mae:
            best_mae, best_epoch, stale = val_mae, epoch, 0
            torch.save({'model_state_dict': model.state_dict(), 'experiment_name': experiment_name,
                        'use_classification': use_classification, 'use_attention': use_attention,
                        'best_epoch': best_epoch, 'best_validation_mae': best_mae}, ckpt_path)
        else:
            stale += 1
        if stale >= PATIENCE:
            print('Early stopping.')
            break

    pd.DataFrame(history).to_csv(OUTPUT_DIR / f'{experiment_name}_history.csv', index=False)
    print('Best epoch:', best_epoch, '| Best validation MAE:', round(best_mae, 4))
    del model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {'experiment_name': experiment_name, 'checkpoint': str(ckpt_path),
            'best_epoch': best_epoch, 'best_validation_mae': best_mae,
            'history_path': str(OUTPUT_DIR / f'{experiment_name}_history.csv')}

print('Training function defined.')

## Training runs

Four runs on 9,500 videos may take a long time. Use a Kaggle GPU. Start with the baseline and Experiment A if your session time is limited; the saved checkpoint and history files remain in `/kaggle/working/echonet_fg_results` for the current session. Do not evaluate on the test set to make architecture decisions; validation MAE is used for checkpoint selection.

In [ ]:
# CELL 10 — Run the four controlled experiments
experiment_configs = [
    {'experiment_name': 'baseline', 'use_classification': False, 'use_attention': False},
    {'experiment_name': 'experiment_A_classification', 'use_classification': True, 'use_attention': False},
    {'experiment_name': 'experiment_B_attention', 'use_classification': False, 'use_attention': True},
    {'experiment_name': 'experiment_AB_combined', 'use_classification': True, 'use_attention': True},
]

training_results = []
for cfg in experiment_configs:
    # Skip a run only if a checkpoint already exists; set FORCE_RERUN=True to retrain.
    FORCE_RERUN = False
    existing = OUTPUT_DIR / f"{cfg['experiment_name']}_best.pt"
    if existing.exists() and not FORCE_RERUN:
        ckpt = torch.load(existing, map_location='cpu', weights_only=False)
        hist_path = OUTPUT_DIR / f"{cfg['experiment_name']}_history.csv"
        training_results.append({'experiment_name': cfg['experiment_name'], 'checkpoint': str(existing),
                                 'best_epoch': ckpt.get('best_epoch'),
                                 'best_validation_mae': ckpt.get('best_validation_mae'),
                                 'history_path': str(hist_path)})
        print('Using existing checkpoint:', existing)
    else:
        training_results.append(train_experiment(**cfg))

training_summary = pd.DataFrame(training_results)
training_summary.to_csv(OUTPUT_DIR / 'training_summary.csv', index=False)
display(training_summary)

In [ ]:
# CELL 11 — Final held-out test evaluation and saved predictions
# Run this after training choices are finalized. Avoid repeated tuning based on test results.
def evaluate_checkpoint(result):
    ckpt = torch.load(result['checkpoint'], map_location=DEVICE, weights_only=False)
    model = EchoFineGrainedModel(
        use_classification=ckpt['use_classification'],
        use_attention=ckpt['use_attention'],
        pretrained=False
    ).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    y, p, vids, c_true, c_pred = predict_loader(model, test_loader)
    mse = mean_squared_error(y, p)
    row = {
        'experiment': result['experiment_name'], 'test_videos': len(y),
        'MAE': mean_absolute_error(y, p), 'MSE': mse, 'RMSE': float(np.sqrt(mse)),
        'R2': r2_score(y, p),
        'Pearson_r': float(pearsonr(y, p).statistic) if len(y)>1 and np.std(y)>0 and np.std(p)>0 else np.nan,
        'within_5_EF_points_percent': float((np.abs(y-p) <= 5).mean()*100)
    }
    pred_df = pd.DataFrame({'video_id': vids, 'actual_EF': y, 'predicted_EF': p,
                            'absolute_error': np.abs(y-p)})
    pred_df.to_csv(OUTPUT_DIR / f"{result['experiment_name']}_predictions.csv", index=False)
    if c_true:
        row['classification_accuracy'] = accuracy_score(c_true, c_pred)
        row['classification_macro_F1'] = f1_score(c_true, c_pred, average='macro', zero_division=0)
        print('\nClassification report —', result['experiment_name'])
        print(classification_report(c_true, c_pred, labels=[0,1,2],
                                    target_names=['EF < 40','EF 40–50','EF > 50'], zero_division=0))
        print('Confusion matrix (rows=true, columns=predicted):\n', confusion_matrix(c_true, c_pred, labels=[0,1,2]))
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return row

metrics = []
for result in training_results:
    metrics.append(evaluate_checkpoint(result))
comparison = pd.DataFrame(metrics)
comparison.to_csv(OUTPUT_DIR / 'test_metrics_comparison.csv', index=False)
display(comparison.round(4))

In [ ]:
# CELL 12 — Plots, error inspection, and package outputs
for result in training_results:
    hp = Path(result['history_path'])
    if hp.exists():
        hist = pd.read_csv(hp)
        plt.figure(figsize=(7,4))
        plt.plot(hist['epoch'], hist['val_mae'], marker='o')
        plt.xlabel('Epoch'); plt.ylabel('Validation MAE (EF points)')
        plt.title(f"Validation MAE — {result['experiment_name']}")
        plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for result in training_results:
    pth = OUTPUT_DIR / f"{result['experiment_name']}_predictions.csv"
    if not pth.exists(): continue
    pred = pd.read_csv(pth)
    plt.figure(figsize=(5.5,5.5))
    plt.scatter(pred.actual_EF, pred.predicted_EF, alpha=0.35)
    lo = min(pred.actual_EF.min(), pred.predicted_EF.min())
    hi = max(pred.actual_EF.max(), pred.predicted_EF.max())
    plt.plot([lo,hi], [lo,hi], linestyle='--')
    plt.xlabel('Actual EF (%)'); plt.ylabel('Predicted EF (%)')
    plt.title(result['experiment_name']); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()
    print('Largest errors:', result['experiment_name'])
    display(pred.sort_values('absolute_error', ascending=False).head(10))

print('Output files:')
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file(): print(f'{p.name:42s} {p.stat().st_size/1024**2:.2f} MB')
print('Free disk GB:', round(shutil.disk_usage('/kaggle/working').free/1024**3, 2))

## Option C — Segmentation is not included in these four runs

The segmentation-assisted option needs genuine left-ventricle tracing annotations from `VolumeTracings.csv`, accurate frame-number mapping, and masks generated in the original image coordinate system. Do not create fake masks or use EF categories as segmentation labels. First finish the regression/classification/attention comparisons; then build segmentation as a separate, annotation-supervised experiment.